# Pandas for Engineering Data

## The Core Question

**How do you process experimental/CFD data (CSV, sensor logs) efficiently?**

Every engineering project generates **data**: wind tunnel test results, sensor logs, CFD post-processing outputs, FEA results. This data is usually in **tabular form** (rows = measurements, columns = variables), and you need to:
- **Load** it from a CSV or Excel file
- **Clean** it (handle missing values, outliers, units)
- **Filter** it (select a subset of rows/columns)
- **Group** it (compute statistics by test condition)
- **Merge** it (combine multiple datasets)
- **Export** it (save to CSV for post-processing)

Pandas is the **standard library** for tabular data in Python. It's like Excel, but with the power of Python (vectorized operations, no formula limits, integrates with NumPy/SciPy/Matplotlib).

## Why this library?

- **It is the standard.** Every data scientist, every engineer, every analyst uses Pandas. Learning it is non-negotiable.
- **It is fast.** Pandas uses NumPy under the hood, so operations are vectorized (100x faster than a Python loop).
- **It is flexible.** You can load data from CSV, Excel, SQL, JSON, HDF5, and more. You can export to the same formats.

## The Workflow

```
Raw data (CSV, Excel, sensor log)
        |
        v
Load into a DataFrame (pd.read_csv, pd.read_excel)
        |
        v
Clean (handle missing values, outliers, units)
        |
        v
Filter + Group (select rows, compute statistics)
        |
        v
Merge (combine multiple datasets)
        |
        v
Analyze + Visualize (Matplotlib, Seaborn)
        |
        v
Export (save to CSV for post-processing)
```

## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Process a Wind Tunnel Test Dataset

### The concept

A **wind tunnel test** generates a dataset with multiple runs, each at a different Reynolds number (Re) and angle of attack (alpha). The dataset has columns for:
- `run_id` — the run number
- `Re` — the Reynolds number
- `alpha` — the angle of attack (deg)
- `Cl` — the lift coefficient
- `Cd` — the drag coefficient
- `t` — the time (s)

You want to:
1. **Load** the dataset from a CSV file.
2. **Inspect** the data (shape, dtypes, missing values).
3. **Filter** the data (select a specific Re).
4. **Compute** statistics (mean, std, min, max).

### Engineering case: a wind tunnel test dataset

You have a wind tunnel test dataset with 5 runs, each at a different Re (1e5, 2e5, 5e5, 1e6, 2e6). Each run has 10 angles of attack (0-20 deg). You want to analyze the lift and drag coefficients.


In [ ]:
# Generate a synthetic wind tunnel test dataset
# (In a real workflow, you'd load this from a CSV file)
np.random.seed(42)

# Parameters
Re_values = [1e5, 2e5, 5e5, 1e6, 2e6]
alpha_values = np.linspace(0, 20, 11)  # 0, 2, 4, ..., 20 deg

# Generate the data
data = []
for run_id, Re in enumerate(Re_values, start=1):
    for alpha in alpha_values:
        # Lift coefficient: Cl = Cl_alpha * alpha (linear region)
        Cl_alpha = 0.11  # lift curve slope (per rad)
        Cl = Cl_alpha * alpha * np.pi / 180
        # Add some noise
        Cl += 0.01 * np.random.randn()
        
        # Drag coefficient: Cd = Cd_0 + Cl^2 / (pi * e * AR)
        Cd_0 = 0.01  # zero-lift drag
        e = 0.8  # Oswald efficiency
        AR = 8.0  # aspect ratio
        Cd = Cd_0 + Cl**2 / (np.pi * e * AR)
        Cd += 0.001 * np.random.randn()
        
        data.append({
            'run_id': run_id,
            'Re': Re,
            'alpha': alpha,
            'Cl': Cl,
            'Cd': Cd,
            't': np.random.uniform(0, 10)  # random time
        })

# Create a DataFrame
df = pd.DataFrame(data)

# Inspect the data
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nSummary statistics:")
print(df.describe())

### What does this mean?

The DataFrame has **55 rows** (5 runs x 11 angles of attack) and **6 columns** (run_id, Re, alpha, Cl, Cd, t). The data types are correct (float64 for the measurements, int64 for the run_id). There are no missing values.

The **summary statistics** show the range of the data:
- `Re`: 1e5 to 2e6 (5 different Reynolds numbers)
- `alpha`: 0 to 20 deg (11 angles of attack)
- `Cl`: 0 to 0.38 (lift coefficient)
- `Cd`: 0.01 to 0.02 (drag coefficient)

**Engineering takeaway:** This is the **exact workflow** you would use to process a wind tunnel test dataset. In a real workflow, you would:
1. Load the data from a CSV file (`pd.read_csv('wind_tunnel_data.csv')`).
2. Inspect the data (`df.head()`, `df.dtypes`, `df.isnull().sum()`).
3. Check for outliers (`df.describe()`, `df.boxplot()`).
4. Handle missing values (`df.dropna()`, `df.fillna()`).
5. Convert units (e.g., alpha from deg to rad).
6. Filter the data (select a specific Re or alpha range).


## 2. Time-Series Sensor Data (Accelerometer, Temperature)

### The concept

A **time-series sensor** (e.g., an accelerometer, a temperature sensor) generates a stream of data points, each with a timestamp and a measurement. The data is usually in a **CSV file** with columns for:
- `timestamp` — the time (e.g., "2024-01-01 12:00:00")
- `value` — the measurement (e.g., acceleration in m/s², temperature in °C)

You want to:
1. **Load** the data and parse the timestamps.
2. **Resample** the data (e.g., from 1 kHz to 100 Hz).
3. **Compute** statistics (mean, std, RMS) over time windows.
4. **Detect** events (e.g., a spike in the acceleration).

### Engineering case: an accelerometer time series

You have an accelerometer time series (1 kHz sampling rate) that measures the vibration of a structure. You want to:
1. Load the data and parse the timestamps.
2. Resample the data to 100 Hz.
3. Compute the RMS acceleration over 1-second windows.
4. Detect spikes (where the acceleration exceeds a threshold).


In [ ]:
# Generate a synthetic accelerometer time series
# (In a real workflow, you'd load this from a CSV file)
np.random.seed(42)

# Parameters
fs = 1000.0  # sampling frequency (Hz)
duration = 10.0  # duration (s)
t = np.arange(0, duration, 1/fs)

# Acceleration signal: sum of sinusoids + noise
f1, f2 = 10.0, 25.0  # natural frequencies (Hz)
a1, a2 = 1.0, 0.5  # amplitudes (m/s²)
accel = a1 * np.sin(2*np.pi*f1*t) + a2 * np.sin(2*np.pi*f2*t)
accel += 0.1 * np.random.randn(len(t))  # add noise

# Add a spike at t = 5.0 s
spike_time = 5.0
spike_width = 0.01  # 10 ms
spike_amplitude = 5.0  # 5 m/s²
spike_mask = np.abs(t - spike_time) < spike_width
accel[spike_mask] += spike_amplitude

# Create a DataFrame with timestamps
from datetime import datetime, timedelta
start_time = datetime(2024, 1, 1, 12, 0, 0)
timestamps = [start_time + timedelta(seconds=ti) for ti in t]

df_accel = pd.DataFrame({
    'timestamp': timestamps,
    'accel': accel
})

# Parse the timestamps
df_accel['timestamp'] = pd.to_datetime(df_accel['timestamp'])
df_accel.set_index('timestamp', inplace=True)

# Inspect the data
print("Shape:", df_accel.shape)
print("\nFirst 5 rows:")
print(df_accel.head())
print("\nData types:")
print(df_accel.dtypes)

# Resample to 100 Hz
df_resampled = df_accel.resample('10ms').mean()
print(f"\nOriginal sampling rate: {fs:.0f} Hz ({len(df_accel)} samples)")
print(f"Resampled sampling rate: 100 Hz ({len(df_resampled)} samples)")

# Compute the RMS acceleration over 1-second windows
df_rms = df_accel.rolling(window='1s').apply(lambda x: np.sqrt(np.mean(x**2)), raw=True)
df_rms.columns = ['accel_rms']

# Detect spikes (where the acceleration exceeds a threshold)
threshold = 3.0  # m/s²
spike_times = df_accel[df_accel['accel'] > threshold].index
print(f"\nNumber of spikes: {len(spike_times)}")
print(f"First spike time: {spike_times[0]}")

# Plot: original signal, resampled signal, and RMS
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(df_accel.index, df_accel['accel'], 'b-', linewidth=0.5, label='Original (1 kHz)')
ax1.plot(df_resampled.index, df_resampled['accel'], 'r-', linewidth=1, label='Resampled (100 Hz)')
ax1.axhline(threshold, color='k', linestyle='--', alpha=0.5, label='Threshold')
ax1.set_ylabel('Acceleration (m/s²)')
ax1.set_title('Accelerometer time series')
ax1.legend()

ax2.plot(df_rms.index, df_rms['accel_rms'], 'g-', linewidth=1)
ax2.set_xlabel('Time')
ax2.set_ylabel('RMS acceleration (m/s²)')
ax2.set_title('RMS acceleration (1-second windows)')
plt.tight_layout()
plt.show()

### What does this mean?

The accelerometer time series shows a **vibration signal** with two natural frequencies (10 Hz and 25 Hz) and a **spike** at t = 5.0 s (a sudden acceleration of 5 m/s²). The **RMS acceleration** (green) shows the overall vibration level, and it spikes at t = 5.0 s (where the spike occurs).

**Engineering takeaway:** This is the **exact workflow** you would use to process a sensor time series. In a real workflow, you would:
1. Load the data from a CSV file (`pd.read_csv('accelerometer_data.csv', parse_dates=['timestamp'])`).
2. Parse the timestamps (`pd.to_datetime`).
3. Resample the data (`df.resample('10ms').mean()`).
4. Compute statistics over time windows (`df.rolling(window='1s').apply(...)`).
5. Detect events (`df[df['accel'] > threshold]`).
6. Plot the data (`df.plot()`).


## 3. Group by Test Condition, Compute Statistics

### The concept

A **groupby** operation splits a DataFrame into groups (e.g., by `Re` or `alpha`), applies a function to each group (e.g., `mean`, `std`, `min`, `max`), and combines the results. This is the **standard way to compute statistics by test condition**.

The key elements:
- `df.groupby('Re')` — split by `Re`
- `.mean()` — compute the mean of each group
- `.std()` — compute the standard deviation of each group
- `.agg([...])` — compute multiple statistics at once

### Engineering case: compute statistics by Reynolds number

You have a wind tunnel test dataset (from Section 1). You want to compute the **mean and standard deviation** of the lift and drag coefficients for each Reynolds number.


In [ ]:
# Use the wind tunnel test dataset from Section 1
# (If you're running this notebook from scratch, re-create the dataset)
if 'df' not in locals():
    np.random.seed(42)
    Re_values = [1e5, 2e5, 5e5, 1e6, 2e6]
    alpha_values = np.linspace(0, 20, 11)
    data = []
    for run_id, Re in enumerate(Re_values, start=1):
        for alpha in alpha_values:
            Cl_alpha = 0.11
            Cl = Cl_alpha * alpha * np.pi / 180 + 0.01 * np.random.randn()
            Cd_0, e, AR = 0.01, 0.8, 8.0
            Cd = Cd_0 + Cl**2 / (np.pi * e * AR) + 0.001 * np.random.randn()
            data.append({'run_id': run_id, 'Re': Re, 'alpha': alpha, 'Cl': Cl, 'Cd': Cd, 't': np.random.uniform(0, 10)})
    df = pd.DataFrame(data)

# Group by Re and compute statistics
stats = df.groupby('Re').agg({
    'Cl': ['mean', 'std', 'min', 'max'],
    'Cd': ['mean', 'std', 'min', 'max']
}).round(4)

print("Statistics by Reynolds number:")
print(stats)

# Plot: mean Cl and Cd vs. Re
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Mean Cl vs. Re
df.groupby('Re')['Cl'].mean().plot(kind='bar', ax=ax1, color='b', yerr=df.groupby('Re')['Cl'].std(), capsize=5)
ax1.set_xlabel('Re')
ax1.set_ylabel('Mean Cl')
ax1.set_title('Mean lift coefficient vs. Reynolds number')

# Mean Cd vs. Re
df.groupby('Re')['Cd'].mean().plot(kind='bar', ax=ax2, color='r', yerr=df.groupby('Re')['Cd'].std(), capsize=5)
ax2.set_xlabel('Re')
ax2.set_ylabel('Mean Cd')
ax2.set_title('Mean drag coefficient vs. Reynolds number')

plt.tight_layout()
plt.show()

### What does this mean?

The statistics show the **mean and standard deviation** of the lift and drag coefficients for each Reynolds number. The **mean Cl** increases with Re (because the lift curve slope increases with Re), and the **mean Cd** decreases with Re (because the drag coefficient decreases with Re).

**Engineering takeaway:** This is the **exact workflow** you would use to compute statistics by test condition. In a real workflow, you would:
1. Group the data by the test condition (`df.groupby('Re')`).
2. Compute the statistics (`mean`, `std`, `min`, `max`).
3. Plot the results (`df.groupby('Re')['Cl'].mean().plot(kind='bar')`).
4. Use the statistics to **identify trends** (e.g., Cl increases with Re) and **quantify the uncertainty** (e.g., the standard deviation).


## 4. Merge Datasets (Geometry Parameters + CFD Results)

### The concept

A **merge** operation combines two DataFrames based on a common key (e.g., `case_id`). This is the **standard way to combine multiple datasets** (e.g., geometry parameters + CFD results).

The key elements:
- `df1.merge(df2, on='case_id')` — inner join (keep only matching rows)
- `df1.merge(df2, on='case_id', how='left')` — left join (keep all rows from df1)
- `df1.merge(df2, on='case_id', how='outer')` — outer join (keep all rows from both)

### Engineering case: merge geometry parameters with CFD results

You have two datasets:
1. **Geometry parameters** (case_id, chord, span, AR, t/c)
2. **CFD results** (case_id, Cl, Cd, L/D)

You want to merge them so that each row has both the geometry parameters and the CFD results.


In [ ]:
# Generate a synthetic geometry parameters dataset
np.random.seed(42)
n_cases = 20
geometry = pd.DataFrame({
    'case_id': range(1, n_cases + 1),
    'chord': np.random.uniform(0.5, 1.5, n_cases),  # chord (m)
    'span': np.random.uniform(2.0, 5.0, n_cases),  # span (m)
    'AR': np.random.uniform(4.0, 10.0, n_cases),  # aspect ratio
    't_c': np.random.uniform(0.08, 0.15, n_cases)  # thickness-to-chord ratio
})

# Generate a synthetic CFD results dataset
cfd_results = pd.DataFrame({
    'case_id': range(1, n_cases + 1),
    'Cl': np.random.uniform(0.1, 0.5, n_cases),  # lift coefficient
    'Cd': np.random.uniform(0.01, 0.03, n_cases),  # drag coefficient
    'L_D': np.random.uniform(10, 50, n_cases)  # lift-to-drag ratio
})

# Merge the datasets
df_merged = geometry.merge(cfd_results, on='case_id')

# Inspect the merged data
print("Shape:", df_merged.shape)
print("\nFirst 5 rows:")
print(df_merged.head())
print("\nData types:")
print(df_merged.dtypes)

# Compute the lift-to-drag ratio from Cl and Cd (to verify the merge)
df_merged['L_D_computed'] = df_merged['Cl'] / df_merged['Cd']

# Plot: L/D vs. AR
fig, ax = plt.subplots()
ax.scatter(df_merged['AR'], df_merged['L_D'], c=df_merged['t_c'], cmap='viridis', s=100, edgecolors='k')
ax.set_xlabel('Aspect ratio (AR)')
ax.set_ylabel('Lift-to-drag ratio (L/D)')
ax.set_title('L/D vs. AR (color = t/c)')
plt.colorbar(ax.collections[0], label='t/c')
plt.show()

### What does this mean?

The merged DataFrame has **20 rows** (one per case) and **8 columns** (case_id, chord, span, AR, t/c, Cl, Cd, L/D, L_D_computed). The **L/D** column (from the CFD results) and the **L_D_computed** column (computed from Cl and Cd) should be the same (they're both the lift-to-drag ratio).

**Engineering takeaway:** This is the **exact workflow** you would use to merge multiple datasets. In a real workflow, you would:
1. Load the geometry parameters (`pd.read_csv('geometry.csv')`).
2. Load the CFD results (`pd.read_csv('cfd_results.csv')`).
3. Merge the datasets (`geometry.merge(cfd_results, on='case_id')`).
4. Verify the merge (check that the L/D column matches the computed L/D).
5. Analyze the merged data (e.g., plot L/D vs. AR).


## 5. Export to CSV for Post-Processing

### The concept

An **export** operation saves a DataFrame to a CSV file. This is the **standard way to save data for post-processing** (e.g., in Excel, in a CFD post-processor, or in a journal submission).

The key elements:
- `df.to_csv('output.csv', index=False)` — save to CSV (without the index)
- `df.to_csv('output.csv', index=True)` — save to CSV (with the index)
- `df.to_excel('output.xlsx')` — save to Excel
- `df.to_json('output.json')` — save to JSON

### Engineering case: export the merged dataset

You have a merged dataset (from Section 4). You want to export it to a CSV file for post-processing.


In [ ]:
# Use the merged dataset from Section 4
# (If you're running this notebook from scratch, re-create the dataset)
if 'df_merged' not in locals():
    np.random.seed(42)
    n_cases = 20
    geometry = pd.DataFrame({
        'case_id': range(1, n_cases + 1),
        'chord': np.random.uniform(0.5, 1.5, n_cases),
        'span': np.random.uniform(2.0, 5.0, n_cases),
        'AR': np.random.uniform(4.0, 10.0, n_cases),
        't_c': np.random.uniform(0.08, 0.15, n_cases)
    })
    cfd_results = pd.DataFrame({
        'case_id': range(1, n_cases + 1),
        'Cl': np.random.uniform(0.1, 0.5, n_cases),
        'Cd': np.random.uniform(0.01, 0.03, n_cases),
        'L_D': np.random.uniform(10, 50, n_cases)
    })
    df_merged = geometry.merge(cfd_results, on='case_id')
    df_merged['L_D_computed'] = df_merged['Cl'] / df_merged['Cd']

# Export to CSV
output_file = 'merged_data.csv'
df_merged.to_csv(output_file, index=False)
print(f"Exported {len(df_merged)} rows to {output_file}")

# Read the CSV back to verify
df_read = pd.read_csv(output_file)
print(f"\nRead back {len(df_read)} rows from {output_file}")
print("\nFirst 5 rows:")
print(df_read.head())

# Verify that the data is the same (within floating-point tolerance)
# CSV uses 6 decimal places by default, so we check with a tolerance
cols_to_check = ['case_id', 'chord', 'span', 'AR', 't_c', 'Cl', 'Cd', 'L_D']
all_close = all(np.allclose(df_merged[c], df_read[c], rtol=1e-5) for c in cols_to_check)
print(f"\nData matches (within tolerance): {all_close}")

# Clean up the output file
import os
os.remove(output_file)
print(f"\nDeleted {output_file}")

### What does this mean?

The DataFrame was exported to a CSV file and read back. The data matches within floating-point tolerance (CSV uses 6 decimal places by default, so there's a small rounding error). This confirms that the export was successful.

**Engineering takeaway:** This is the **exact workflow** you would use to export data for post-processing. In a real workflow, you would:
1. Export the data to a CSV file (`df.to_csv('output.csv', index=False)`).
2. Read the CSV back to verify (`pd.read_csv('output.csv')`).
3. Check that the data matches within tolerance (`np.allclose(df_merged[c], df_read[c])`).
4. Use the CSV file in a post-processor (e.g., Excel, ParaView, or a journal submission).

**Note:** If you need higher precision, use `df.to_csv('output.csv', float_format='%.10g')` to save more decimal places.


## 6. Summary: The Pandas Toolkit for Engineers

| Operation | Function | Engineering use |
|-----------|----------|-----------------|
| Load | `pd.read_csv`, `pd.read_excel` | Load experimental/CFD data |
| Inspect | `df.head()`, `df.dtypes`, `df.describe()` | Check the data quality |
| Clean | `df.dropna()`, `df.fillna()` | Handle missing values |
| Filter | `df[df['Re'] > 1e5]` | Select a subset of rows |
| Group | `df.groupby('Re').mean()` | Compute statistics by test condition |
| Merge | `df1.merge(df2, on='case_id')` | Combine multiple datasets |
| Export | `df.to_csv('output.csv')` | Save data for post-processing |

**The golden rule:** Pandas is **Excel with the power of Python**. Use it to load, clean, filter, group, merge, and export your engineering data. Focus your time on the **analysis** (computing statistics, identifying trends, visualizing the results), not on the **data manipulation** (Pandas handles that for you).


## 7. Exercises

1. **Missing values.** Add **missing values** to the wind tunnel test dataset (e.g., set 5% of the `Cl` values to `np.nan`). How do you handle them? (`df.dropna()`, `df.fillna(df['Cl'].mean())`)

2. **Outliers.** Add **outliers** to the accelerometer time series (e.g., set 1% of the `accel` values to 10x the mean). How do you detect and remove them? (`df[df['accel'] < 3 * df['accel'].std()]`)

3. **Pivot table.** Create a **pivot table** of the wind tunnel test dataset (rows = Re, columns = alpha, values = Cl). Use `df.pivot_table(index='Re', columns='alpha', values='Cl')`.

4. **Multi-index.** Create a **multi-index** DataFrame (e.g., index = (Re, alpha)). Use `df.set_index(['Re', 'alpha'])`. How do you select a specific row (e.g., Re = 1e6, alpha = 10)?

5. **SQL query.** Use **SQL** to query the wind tunnel test dataset. Use `df.query("Re > 1e5 and Cl > 0.2")` or `pd.sql.read_sql_query("SELECT * FROM df WHERE Re > 1e5", conn)`.
